AI was used in this file

In [1]:
import os
import re
import random
from pathlib import Path
from collections import Counter, defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from model import codeRes18SE

# ResNet18 + SE attention experiment
model_name = "resnet18_se"

root = Path("/Users/xander/3830prdoject/MVTecStyle/my_category")
batch_size = 16
epochs = 10
lr = 1e-4
seed = 42
img_size = 224

random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)
print("dataset root:", root)


device: mps
dataset root: /Users/xander/3830prdoject/MVTecStyle/my_category


In [2]:
# 数据集已经离线增强过，这里不再使用随机翻转、旋转、颜色扰动等在线增强。
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [3]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
group_suffix = re.compile(r"_(?:orig|aug\d+)$", re.IGNORECASE)

# 类别映射：0 是正品，1 是次品
class_to_idx = {"good": 0, "defective": 1}


def check_folder(folder):
    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")


def group_key(path):
    return group_suffix.sub("", path.stem)


def collect_groups(folder, label):
    check_folder(folder)
    groups = defaultdict(list)
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() in image_exts:
            groups[group_key(path)].append((path, label, group_key(path)))
    return groups


def merge_groups(*group_dicts):
    merged = defaultdict(list)
    for group_dict in group_dicts:
        for key, samples in group_dict.items():
            merged[key].extend(samples)
    return merged


def split_group_dict(groups, train_ratio=0.7, val_ratio=0.15):
    keys = list(groups.keys())
    random.shuffle(keys)

    n = len(keys)
    n_train = round(n * train_ratio)
    n_val = round(n * val_ratio)

    train_keys = keys[:n_train]
    val_keys = keys[n_train:n_train + n_val]
    test_keys = keys[n_train + n_val:]

    def flatten(selected_keys):
        samples = []
        for key in selected_keys:
            samples.extend(groups[key])
        return samples

    return flatten(train_keys), flatten(val_keys), flatten(test_keys)


class ProductDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label, _ = self.samples[index]
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        return image, label


good_groups = merge_groups(
    collect_groups(root / "train" / "good", class_to_idx["good"]),
    collect_groups(root / "test" / "good", class_to_idx["good"])
)

defective_groups = collect_groups(root / "test" / "defective", class_to_idx["defective"])

good_train, good_val, good_test = split_group_dict(good_groups)
defect_train, defect_val, defect_test = split_group_dict(defective_groups)

train_samples = good_train + defect_train
val_samples = good_val + defect_val
test_samples = good_test + defect_test

random.shuffle(train_samples)

# 检查同一原图 group 是否跨 train/val/test，避免增强图泄漏。
seen = defaultdict(set)
for split_name, samples in [("train", train_samples), ("val", val_samples), ("test", test_samples)]:
    for _, label, key in samples:
        seen[(label, key)].add(split_name)
leaked_groups = [item for item, split_names in seen.items() if len(split_names) > 1]
if leaked_groups:
    raise RuntimeError(f"Data leakage detected: {len(leaked_groups)} leaked groups")

print("class_to_idx:", class_to_idx)
print("train:", len(train_samples), Counter(label for _, label, _ in train_samples))
print("val:", len(val_samples), Counter(label for _, label, _ in val_samples))
print("test:", len(test_samples), Counter(label for _, label, _ in test_samples))
print("leaked_groups:", len(leaked_groups))


class_to_idx: {'good': 0, 'defective': 1}
train: 2992 Counter({1: 2437, 0: 555})
val: 613 Counter({1: 478, 0: 135})
test: 631 Counter({1: 495, 0: 136})
leaked_groups: 0


In [4]:
train_dataset = ProductDataset(train_samples, transform)
val_dataset = ProductDataset(val_samples, transform)
test_dataset = ProductDataset(test_samples, transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [5]:
model = codeRes18SE(num_classes=2)
model = model.to(device)

# 类别不均衡，次品和正品数量差很多，所以加 class weight。
train_counts = Counter(label for _, label, _ in train_samples)
total = len(train_samples)
class_weights = torch.tensor([
    total / (2 * train_counts[0]),
    total / (2 * train_counts[1])
], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print("model:", model_name)
print("class_weights:", class_weights.detach().cpu().numpy())


model: resnet18_se
class_weights: [2.6954956 0.6138695]


In [6]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    tp = fp = tn = fn = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            tn += ((preds == 0) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()

    accuracy = correct / total
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return {
        "loss": total_loss / total,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

In [7]:
save_dir = Path("runs") / model_name
save_dir.mkdir(parents=True, exist_ok=True)
best_path = save_dir / "best_model.pth"

best_f1 = -1.0

for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        train_loss += loss.item() * labels.size(0)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss / train_total
    train_acc = train_correct / train_total
    val_result = evaluate(model, val_loader)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"train_loss={train_loss:.4f} "
        f"train_acc={train_acc:.4f} "
        f"val_acc={val_result['accuracy']:.4f} "
        f"val_precision={val_result['precision']:.4f} "
        f"val_recall={val_result['recall']:.4f} "
        f"val_f1={val_result['f1']:.4f}"
    )

    if val_result["f1"] > best_f1:
        best_f1 = val_result["f1"]
        torch.save(model.state_dict(), best_path)
        print("saved:", best_path)


Epoch [1/10] train_loss=0.2702 train_acc=0.8817 val_acc=0.9299 val_precision=0.9823 val_recall=0.9268 val_f1=0.9537
saved: runs/resnet18_se/best_model.pth
Epoch [2/10] train_loss=0.0801 train_acc=0.9723 val_acc=0.9592 val_precision=0.9789 val_recall=0.9686 val_f1=0.9737
saved: runs/resnet18_se/best_model.pth
Epoch [3/10] train_loss=0.0510 train_acc=0.9843 val_acc=0.9788 val_precision=0.9794 val_recall=0.9937 val_f1=0.9865
saved: runs/resnet18_se/best_model.pth
Epoch [4/10] train_loss=0.0291 train_acc=0.9910 val_acc=0.9576 val_precision=0.9728 val_recall=0.9728 val_f1=0.9728
Epoch [5/10] train_loss=0.0625 train_acc=0.9773 val_acc=0.8728 val_precision=0.9808 val_recall=0.8536 val_f1=0.9128
Epoch [6/10] train_loss=0.0428 train_acc=0.9853 val_acc=0.9511 val_precision=0.9726 val_recall=0.9644 val_f1=0.9685
Epoch [7/10] train_loss=0.0297 train_acc=0.9900 val_acc=0.9217 val_precision=0.9799 val_recall=0.9184 val_f1=0.9482
Epoch [8/10] train_loss=0.0179 train_acc=0.9947 val_acc=0.9837 val_prec

In [8]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_result = evaluate(model, test_loader)

print("test result for defective class:")
print("accuracy:", test_result["accuracy"])
print("precision_defective:", test_result["precision"])
print("recall_defective:", test_result["recall"])
print("f1_defective:", test_result["f1"])
print("confusion matrix")
print("              pred_good  pred_defective")
print("true_good    ", test_result["tn"], "        ", test_result["fp"])
print("true_defect  ", test_result["fn"], "        ", test_result["tp"])


/var/folders/_d/87160cl95kz9qrx_337cfyzr0000gn/T/ipykernel_65713/1809008037.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, m

test result for defective class:
accuracy: 0.9889064976228209
precision_defective: 0.9899598393375509
recall_defective: 0.9959595959394756
f1_defective: 0.9929506495621212
confusion matrix
              pred_good  pred_defective
true_good     131          5
true_defect   2          493
